In [101]:
import pandas as pd
import numpy as np
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

LOOKBACK = 10 # গত 10 দিন দেখে পরের দিন predict করবে


In [102]:
data = pd.read_csv('../bitcoin_data/raw/btc_daily_clean.csv')
data.head()

,date,open,high,low,close,volume
0,2014-09-17,465.864014,468.174011,452.421997,457.334015,21056800
1,2014-09-18,456.859985,456.859985,413.104004,424.440002,34483200
2,2014-09-19,424.102997,427.834991,384.532013,394.795990,37919700
3,2014-09-20,394.673004,423.295990,389.882996,408.903992,36863600
4,2014-09-21,408.084991,412.425995,393.181000,398.821014,26580100


In [103]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 4298 entries, 0 to 4297
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    4298 non-null   str    
 1   open    4298 non-null   float64
 2   high    4298 non-null   float64
 3   low     4298 non-null   float64
 4   close   4298 non-null   float64
 5   volume  4298 non-null   int64  
dtypes: float64(4), int64(1), str(1)
memory usage: 201.6 KB


In [104]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 4298 entries, 0 to 4297
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    4298 non-null   str    
 1   open    4298 non-null   float64
 2   high    4298 non-null   float64
 3   low     4298 non-null   float64
 4   close   4298 non-null   float64
 5   volume  4298 non-null   int64  
dtypes: float64(4), int64(1), str(1)
memory usage: 201.6 KB


In [105]:
df = data.copy()

### Feature Engineering

In [106]:
def compute_rsi(close, window=14):
    delta = close.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(alpha=1 / window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / window, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    return rsi


def compute_atr(df, window=14):
    high_low = df["high"] - df["low"]
    high_close = (df["high"] - df["close"].shift(1)).abs()
    low_close = (df["low"] - df["close"].shift(1)).abs()

    true_range = pd.concat(
        [high_low, high_close, low_close],
        axis=1
    ).max(axis=1)

    atr = true_range.rolling(window).mean()

    return atr


def rolling_zscore(series, window):
    mean = series.rolling(window).mean()
    std = series.rolling(window).std()

    return (series - mean) / std

In [107]:
def create_btc_indicators(df):
    df = df.copy()
    df = df.sort_index()

    # ----------------------------
    # Price return indicators
    # ----------------------------
    df["log_close"] = np.log(df["close"])

    for h in [1, 2, 3, 5, 7, 14, 21, 30]:
        df[f"log_return_{h}d"] = df["log_close"].diff(h)
        df[f"pct_return_{h}d"] = df["close"].pct_change(h)

    # ----------------------------
    # Moving averages
    # ----------------------------
    for window in [7, 14, 20, 30, 50, 100, 200]:
        df[f"sma_{window}"] = df["close"].rolling(window).mean()
        df[f"ema_{window}"] = df["close"].ewm(span=window, adjust=False).mean()

        df[f"close_to_sma_{window}"] = df["close"] / df[f"sma_{window}"] - 1
        df[f"close_to_ema_{window}"] = df["close"] / df[f"ema_{window}"] - 1

        df[f"sma_{window}_slope"] = df[f"sma_{window}"].pct_change(5)
        df[f"ema_{window}_slope"] = df[f"ema_{window}"].pct_change(5)

    # Moving average crossover indicators
    df["sma_20_above_50"] = (df["sma_20"] > df["sma_50"]).astype(int)
    df["sma_50_above_200"] = (df["sma_50"] > df["sma_200"]).astype(int)
    df["ema_20_above_50"] = (df["ema_20"] > df["ema_50"]).astype(int)

    # ----------------------------
    # RSI
    # ----------------------------
    df["rsi_14"] = compute_rsi(df["close"], window=14)

    df["rsi_overbought"] = (df["rsi_14"] > 70).astype(int)
    df["rsi_oversold"] = (df["rsi_14"] < 30).astype(int)

    # ----------------------------
    # MACD
    # ----------------------------
    ema_12 = df["close"].ewm(span=12, adjust=False).mean()
    ema_26 = df["close"].ewm(span=26, adjust=False).mean()

    df["macd"] = ema_12 - ema_26
    df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
    df["macd_hist"] = df["macd"] - df["macd_signal"]

    df["macd_bullish"] = (df["macd"] > df["macd_signal"]).astype(int)

    # ----------------------------
    # Volatility indicators
    # ----------------------------
    for window in [7, 14, 21, 30, 60, 90]:
        df[f"realized_vol_{window}d"] = df["log_return_1d"].rolling(window).std()
        df[f"realized_vol_{window}d_ann"] = df[f"realized_vol_{window}d"] * np.sqrt(365)

    df["atr_14"] = compute_atr(df, window=14)
    df["atr_pct_14"] = df["atr_14"] / df["close"]

    df["daily_range"] = df["high"] - df["low"]
    df["daily_range_pct"] = df["daily_range"] / df["close"]

    df["close_position_in_range"] = (
        (df["close"] - df["low"]) / (df["high"] - df["low"])
    )

    # ----------------------------
    # Bollinger Bands
    # ----------------------------
    bb_window = 20

    df["bb_middle"] = df["close"].rolling(bb_window).mean()
    bb_std = df["close"].rolling(bb_window).std()

    df["bb_upper"] = df["bb_middle"] + 2 * bb_std
    df["bb_lower"] = df["bb_middle"] - 2 * bb_std

    df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / df["bb_middle"]

    df["bb_percent_b"] = (
        (df["close"] - df["bb_lower"]) /
        (df["bb_upper"] - df["bb_lower"])
    )

    df["price_above_bb_upper"] = (df["close"] > df["bb_upper"]).astype(int)
    df["price_below_bb_lower"] = (df["close"] < df["bb_lower"]).astype(int)

    # ----------------------------
    # Drawdown and breakout indicators
    # ----------------------------
    for window in [30, 60, 90, 180, 365]:
        rolling_high = df["close"].rolling(window).max()
        rolling_low = df["close"].rolling(window).min()

        df[f"drawdown_from_{window}d_high"] = df["close"] / rolling_high - 1
        df[f"distance_from_{window}d_low"] = df["close"] / rolling_low - 1

        df[f"breakout_{window}d_high"] = (
            df["close"] >= rolling_high.shift(1)
        ).astype(int)

        df[f"breakdown_{window}d_low"] = (
            df["close"] <= rolling_low.shift(1)
        ).astype(int)

    # ----------------------------
    # Volume indicators
    # ----------------------------
    df["volume_log"] = np.log1p(df["volume"])

    for window in [7, 14, 30, 60, 90]:
        df[f"volume_sma_{window}"] = df["volume"].rolling(window).mean()
        df[f"volume_z_{window}d"] = rolling_zscore(df["volume_log"], window)
        df[f"volume_change_{window}d"] = df["volume"].pct_change(window)

    df["volume_above_30d_avg"] = (
        df["volume"] > df["volume_sma_30"]
    ).astype(int)

    df["volume_to_volatility"] = (
        df["volume_z_30d"] / df["realized_vol_30d"]
    )

    df = df.replace([np.inf, -np.inf], np.nan)

    return df

In [108]:
new_df = create_btc_indicators(df)

print(new_df.tail())
print(new_df.shape)

            date          open          high           low         close  \
4293  2026-06-19  62897.515625  63568.222656  62275.582031  63540.835938   
4294  2026-06-20  63535.808594  64307.136719  63149.660156  64239.675781   
4295  2026-06-21  64242.324219  64506.578125  63221.000000  63237.539062   
4296  2026-06-22  63240.789062  65544.000000  63233.531250  63952.105469   
4297  2026-06-24  63951.507812  64163.449219  61990.453125  62695.230469   

           volume  log_close  log_return_1d  pct_return_1d  log_return_2d  \
4293  22361931660  11.059438       0.010193       0.010245      -0.013717   
4294  17486384117  11.070376       0.010938       0.010998       0.021131   
4295  15739619591  11.054653      -0.015723      -0.015600      -0.004785   
4296  26561496819  11.065890       0.011236       0.011300      -0.004487   
4297  29468542976  11.046041      -0.019849      -0.019653      -0.008613   

      ...  volume_z_30d  volume_change_30d  volume_sma_60  volume_z_60d  \
4293 

/var/folders/g2/z2xpfp7x5hvgkf6rx1kgrx4w0000gn/T/ipykernel_51125/3156723235.py:98: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"distance_from_{window}d_low"] = df["close"] / rolling_low - 1
/var/folders/g2/z2xpfp7x5hvgkf6rx1kgrx4w0000gn/T/ipykernel_51125/3156723235.py:100: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"breakout_{window}d_high"] = (
/var/folders/g2/z2xpfp7x5hvgkf6rx1kgrx4w0000gn/T/ipykernel_51125/3156723235.py:104: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of callin

In [109]:
# date কে index বানান
new_df = new_df.set_index('date')
new_df.index = pd.to_datetime(new_df.index)

print(new_df.index[:3])
print(new_df.index.dtype)

DatetimeIndex(['2014-09-17', '2014-09-18', '2014-09-19'], dtype='datetime64[us]', name='date', freq=None)
datetime64[us]


In [110]:
new_df_clean = new_df.dropna()

print(new_df.tail())
print(new_df.shape)

                    open          high           low         close  \
date                                                                 
2026-06-19  62897.515625  63568.222656  62275.582031  63540.835938   
2026-06-20  63535.808594  64307.136719  63149.660156  64239.675781   
2026-06-21  64242.324219  64506.578125  63221.000000  63237.539062   
2026-06-22  63240.789062  65544.000000  63233.531250  63952.105469   
2026-06-24  63951.507812  64163.449219  61990.453125  62695.230469   

                 volume  log_close  log_return_1d  pct_return_1d  \
date                                                               
2026-06-19  22361931660  11.059438       0.010193       0.010245   
2026-06-20  17486384117  11.070376       0.010938       0.010998   
2026-06-21  15739619591  11.054653      -0.015723      -0.015600   
2026-06-22  26561496819  11.065890       0.011236       0.011300   
2026-06-24  29468542976  11.046041      -0.019849      -0.019653   

            log_return_2d  pct_r

In [111]:
FEATURE_COLS = [
    # === Returns (relative, scale-free) ===
    'log_return_1d', 'pct_return_1d',
    'log_return_2d', 'pct_return_2d',
    'log_return_3d', 'pct_return_3d',
    'log_return_5d', 'pct_return_5d',
    'log_return_7d', 'pct_return_7d',
    'log_return_14d','pct_return_14d',
    'log_return_21d','pct_return_21d',
    'log_return_30d','pct_return_30d',

    # === MA relative distance (price নয়, distance) ===
    'close_to_sma_7',  'close_to_ema_7',  'sma_7_slope',  'ema_7_slope',
    'close_to_sma_14', 'close_to_ema_14', 'sma_14_slope', 'ema_14_slope',
    'close_to_sma_20', 'close_to_ema_20', 'sma_20_slope', 'ema_20_slope',
    'close_to_sma_50', 'close_to_ema_50', 'sma_50_slope', 'ema_50_slope',
    'close_to_sma_100','close_to_ema_100',
    'close_to_sma_200','close_to_ema_200',

    # === MA Crossover signals ===
    'sma_20_above_50', 'sma_50_above_200', 'ema_20_above_50',

    # === Momentum ===
    'rsi_14', 'rsi_overbought', 'rsi_oversold',
    'macd', 'macd_signal', 'macd_hist', 'macd_bullish',

    # === Volatility ===
    'realized_vol_7d',  'realized_vol_14d', 'realized_vol_21d',
    'realized_vol_30d', 'realized_vol_60d',
    'atr_pct_14',           # atr_14 বাদ — absolute price
    'daily_range_pct',      # daily_range বাদ — absolute price
    'close_position_in_range',

    # === Bollinger Bands (relative only) ===
    'bb_width', 'bb_percent_b',
    'price_above_bb_upper', 'price_below_bb_lower',

    # === Drawdown / Breakout ===
    'drawdown_from_30d_high',  'distance_from_30d_low',
    'breakout_30d_high',       'breakdown_30d_low',
    'drawdown_from_60d_high',  'distance_from_60d_low',
    'drawdown_from_90d_high',  'distance_from_90d_low',
    'breakout_90d_high',       'breakdown_90d_low',
    'drawdown_from_180d_high', 'distance_from_180d_low',
    'drawdown_from_365d_high', 'distance_from_365d_low',

    # === Volume (relative only) ===
    'volume_z_7d',   'volume_change_7d',
    'volume_z_14d',  'volume_change_14d',
    'volume_z_30d',  'volume_change_30d',
    'volume_z_60d',  'volume_change_60d',
    'volume_z_90d',  'volume_change_90d',
    'volume_above_30d_avg',
    'volume_to_volatility',
]

print(f"Total features: {len(FEATURE_COLS)}")

# নিশ্চিত করুন সব column df-এ আছে
missing = [c for c in FEATURE_COLS if c not in new_df.columns]
print(f"Missing columns: {missing}")

Total features: 84
Missing columns: []


In [112]:
# ধাপ ১: আগে target তৈরি করুন
new_df['target'] = new_df['close'].pct_change().shift(-1) * 100
new_df.dropna(subset=['target'], inplace=True)

# ধাপ ২: তারপর FEATURE_COLS বানান
exclude_cols = [
    'date', 'open', 'high', 'low', 'close', 'volume',
    'log_close',
    'sma_7',  'ema_7',  'sma_14', 'ema_14',
    'sma_20', 'ema_20', 'sma_30', 'ema_30',
    'sma_50', 'ema_50', 'sma_100','ema_100', 'sma_200','ema_200',
    'bb_middle', 'bb_upper', 'bb_lower',
    'atr_14', 'daily_range',
    'volume_log', 'volume_sma_7',  'volume_sma_14',
    'volume_sma_30', 'volume_sma_60', 'volume_sma_90',
    'realized_vol_7d_ann',  'realized_vol_14d_ann',
    'realized_vol_21d_ann', 'realized_vol_30d_ann',
    'realized_vol_60d_ann', 'realized_vol_90d_ann',
    'target',
]
FEATURE_COLS = [col for col in new_df.columns if col not in exclude_cols]
print(f"Total features: {len(FEATURE_COLS)}")


Total features: 99


In [113]:
# Scale
scaler     = StandardScaler()
X_all      = new_df[FEATURE_COLS]
y_all      = new_df['target']
n_train    = len(train)

X_scaled   = scaler.fit_transform(X_all)

In [114]:
# Sequence তৈরি
def make_sequences(X, y, lookback):
    Xs, ys = [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i-lookback:i])
        ys.append(y.iloc[i])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = make_sequences(X_scaled, y_all, LOOKBACK)


In [115]:
# ধাপ ১: NaN rows সরিয়ে দিন
new_df_clean = new_df.dropna(subset=FEATURE_COLS)
print(f"আগে: {len(new_df)} rows → পরে: {len(new_df_clean)} rows")
print("NaN remaining:", new_df_clean[FEATURE_COLS].isnull().sum().sum())

# ধাপ ২: আবার split
split_date = pd.Timestamp("2025-01-01")
train      = new_df_clean[new_df_clean.index < split_date]
test       = new_df_clean[new_df_clean.index >= split_date]
print(f"Train: {len(train)} rows | {train.index[0].date()} → {train.index[-1].date()}")
print(f"Test : {len(test)} rows  | {test.index[0].date()} → {test.index[-1].date()}")

# ধাপ ৩: Scale
X_all    = new_df_clean[FEATURE_COLS]
y_all    = new_df_clean['target']
n_train  = len(train)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_all)

# ধাপ ৪: Sequence
X_seq, y_seq = make_sequences(X_scaled, y_all, LOOKBACK)

split        = n_train - LOOKBACK
X_train_seq  = X_seq[:split]
X_test_seq   = X_seq[split:]
y_train_seq  = y_seq[:split]
y_test_seq   = y_seq[split:]

print(f"\nX_train_seq : {X_train_seq.shape}")
print(f"X_test_seq  : {X_test_seq.shape}")
print(f"NaN in X_seq: {np.isnan(X_seq).sum()}")

আগে: 4297 rows → পরে: 3933 rows
NaN remaining: 0
Train: 3395 rows | 2015-09-16 → 2024-12-31
Test : 538 rows  | 2025-01-01 → 2026-06-22

X_train_seq : (3385, 10, 99)
X_test_seq  : (538, 10, 99)
NaN in X_seq: 0


In [116]:
# NaN check
print("X NaN count:", np.isnan(X_scaled).sum())
print("y NaN count:", np.isnan(y_all.values).sum())

# Inf check
print("X Inf count:", np.isinf(X_scaled).sum())
print("y Inf count:", np.isinf(y_all.values).sum())

# কোন column-এ NaN আছে
nan_cols = X_all.columns[X_all.isnull().any()].tolist()
print(f"NaN columns: {nan_cols}")

X NaN count: 0
y NaN count: 0
X Inf count: 0
y Inf count: 0
NaN columns: []


In [117]:
# Model
model = Sequential([
    LSTM(32, return_sequences=False, input_shape=(LOOKBACK, len(FEATURE_COLS))),
    Dropout(0.3),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')
model.summary()

# Train
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history = model.fit(
    X_train_seq, y_train_seq,
    validation_split=0.1,
    epochs=100,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1,
)



/Users/rajibul/Downloads/Data-Science-Projects/Financial_Market_Crypto/crypto/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


ValueError: Input 0 with name 'None' of layer 'lstm_9' is incompatible with the layer: expected ndim=3, found ndim=2. Full shape received: (None, 32)

In [ ]:
# Evaluate
preds      = model.predict(X_test_seq).flatten()
mae        = mean_absolute_error(y_test_seq, preds)
rmse       = np.sqrt(mean_squared_error(y_test_seq, preds))
r2         = r2_score(y_test_seq, preds)
dir_acc    = ((preds > 0) == (y_test_seq > 0)).mean() * 100

print(f"\nMAE           : {mae:.4f}%")
print(f"RMSE          : {rmse:.4f}%")
print(f"R²            : {r2:.4f}")
print(f"Direction Acc : {dir_acc:.2f}%")


17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step

MAE           : 1.6653%
RMSE          : 2.3603%
R²            : -0.0222
Direction Acc : 48.67%


In [118]:
df.index

RangeIndex(start=0, stop=4298, step=1)

In [119]:
new_df.index

DatetimeIndex(['2014-09-17', '2014-09-18', '2014-09-19', '2014-09-20',
               '2014-09-21', '2014-09-22', '2014-09-23', '2014-09-24',
               '2014-09-25', '2014-09-26',
               ...
               '2026-06-13', '2026-06-14', '2026-06-15', '2026-06-16',
               '2026-06-17', '2026-06-18', '2026-06-19', '2026-06-20',
               '2026-06-21', '2026-06-22'],
              dtype='datetime64[us]', name='date', length=4297, freq=None)

In [120]:
new_df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 4297 entries, 2014-09-17 to 2026-06-22
Columns: 137 entries, open to target
dtypes: float64(117), int64(20)
memory usage: 4.5 MB
